# MIMIC-IV ICD Coder â€” Exploratory Data Analysis

**Owner:** Nancy Tanaka  |  **Dataset:** MIMIC-IV-Note v2.2 + MIMIC-IV v3.1 Hosp

## Purpose

Validate data quality and inform modeling decisions BEFORE building Silver/Gold and training the baseline. Outputs of this notebook feed directly into `reports/eda_report.md` and `reports/data_card.md`.

## Prerequisites

Run `mic ingest --config configs/dev.yml` first so Bronze Parquet exists at `data/bronze/`. This notebook reads from there â€” it does NOT re-parse the gzipped CSVs.

## Decisions this EDA informs

- **Top-K labels** â€” top-50 the right choice, or should we shift to top-100 or top-25?
- **Token truncation** â€” how much information is lost at 512 tokens? Does Longformer (4K) matter?
- **Min token filter** â€” 100 tokens too aggressive? Too lenient?
- **ICD version filter** â€” what fraction of admissions is ICD-10 only?
- **Patient-level vs admission-level splits** â€” is the skew patient-heavy?
- **Join losses** â€” how many admissions survive the notes âˆ© diagnoses intersection?

In [ ]:
# Quick sanity check
from mimic_icd_coder import eda

print("eda module loaded:", dir(eda)[:5])


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from mimic_icd_coder import eda

BRONZE = Path("../data/bronze")
FIGURES = Path("../reports/figures")
FIGURES.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 50)
plt.rcParams["figure.dpi"] = 110


In [ ]:
# Load Bronze Parquet â€” fast, typed reads
notes = pd.read_parquet(BRONZE / "discharge_notes.parquet")
diagnoses = pd.read_parquet(BRONZE / "diagnoses_icd.parquet")
admissions = pd.read_parquet(BRONZE / "admissions.parquet")
patients = pd.read_parquet(BRONZE / "patients.parquet")
icd_diagnoses = pd.read_parquet(BRONZE / "d_icd_diagnoses.parquet")

tables = {
    "notes": notes,
    "diagnoses": diagnoses,
    "admissions": admissions,
    "patients": patients,
    "icd_diagnoses": icd_diagnoses,
}

print({k: f"{len(v):,} rows" for k, v in tables.items()})

## 1. Volumetrics â€” do we have what we think we have?

Sanity check row counts and memory footprint. Expected from docs:

| Table | Expected N |
|---|---|
| notes | 331,793 |
| diagnoses | 6,364,488 |
| admissions | 546,028 |
| patients | 364,627 |
| icd_diagnoses | 112,107 

In [ ]:
vol = eda.summarize_volumetrics(tables)
vol

In [ ]:
# Null rates â€” any surprises?
for name, df in tables.items():
    print(f"\n-- {name} --")
    display(eda.null_rate_by_column(df, name).head(10))

In [ ]:
# Date ranges â€” MIMIC-IV spans 2008-2019 (de-identified date shift)
for col in ["charttime", "storetime"]:
    if col in notes.columns:
        print(f"notes.{col}:", eda.date_range(notes, col))
for col in ["admittime", "dischtime"]:
    print(f"admissions.{col}:", eda.date_range(admissions, col))

## 2. Note types

MIMIC-IV-Note v2.2 contains `DS` (discharge summary) and `radiology` note types. We only care about `DS` for this project. Confirm the distribution.

In [ ]:
nt = eda.note_type_distribution(notes)
nt

## 3. Token & character length

Critical for choosing between Bio_ClinicalBERT (512-token cap) and Clinical-Longformer (4K). If p90 > 512, we lose a lot at BERT cap.

In [ ]:
notes_lens = eda.compute_lengths(notes)
pct = eda.length_percentiles(notes_lens)
pct

In [ ]:
# Truncation impact at BERT vs Longformer caps
for cap in [512, 1024, 2048, 4096]:
    r = eda.bert_truncation_impact(notes_lens, max_tokens=cap)
    print(r)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
eda.plot_length_distribution(notes_lens, ax1, column="n_tokens", log_y=True)
eda.plot_length_distribution(notes_lens, ax2, column="n_chars", log_y=True)
plt.tight_layout()
plt.savefig(FIGURES / "length_distribution.png", bbox_inches="tight")
plt.show()

## 4. De-identification marker density

MIMIC uses `___` (underscores) for redacted spans â€” names, dates, IDs. Too much redaction hurts model signal. Too little means encodings may leak PHI.

In [ ]:
deid = eda.deid_marker_stats(notes, sample_size=10_000)
deid

## 5. Duplicate analysis

Are there admissions with multiple notes? What's the note_seq pattern?

In [ ]:
eda.note_duplication_summary(notes)

## 6. ICD version split

MIMIC-IV spans the ICD-9 â†’ ICD-10 transition (Oct 2015). Our cohort filter is ICD-10 only â€” does that kill the project?

In [ ]:
ver = eda.icd_version_distribution(diagnoses)
ver

In [ ]:
# By year â€” visualize the ICD-9 â†’ ICD-10 transition
ver_year = eda.icd_version_distribution(diagnoses, admissions)
pivot = ver_year.pivot(index="year", columns="icd_version", values="n_codes").fillna(0)
fig, ax = plt.subplots(figsize=(10, 4))
pivot.plot(kind="area", stacked=True, ax=ax, colormap="viridis")
ax.set_ylabel("# ICD codes assigned")
ax.set_title("ICD-9 â†’ ICD-10 transition by year")
plt.tight_layout()
plt.savefig(FIGURES / "icd_version_by_year.png", bbox_inches="tight")
plt.show()

## 7. ICD-10 code frequency â€” top-K coverage curve

This is THE chart for choosing `top_k_labels`. If top-50 covers 80% of admissions, great. If it only covers 30%, we rethink.

In [ ]:
freq = eda.icd_frequency(diagnoses, version=10)
print(f"Total distinct ICD-10 codes: {len(freq):,}")
print("\nTop 20 most-assigned codes:")
freq.head(20)

In [ ]:
cov = eda.top_k_coverage(diagnoses, k_list=(10, 25, 50, 100, 250, 500, 1000, 2500))
cov

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
eda.plot_icd_frequency_curve(freq, ax1)
eda.plot_top_k_coverage(cov, ax2)
plt.tight_layout()
plt.savefig(FIGURES / "icd_frequency_and_coverage.png", bbox_inches="tight")
plt.show()

In [ ]:
# Table B3 â€” Top-50 ICD-10 codes with clinical descriptions
# Uses icd_diagnoses already loaded in the Bronze read cell at the top.
# ICD formatter lives in mimic_icd_coder.eda (imported as `eda` in cell 2).

d_icd10 = icd_diagnoses.loc[icd_diagnoses["icd_version"] == 10, ["icd_code", "long_title"]].copy()
d_icd10["icd_code"] = d_icd10["icd_code"].astype(str).str.strip().str.upper()
d_icd10 = d_icd10.drop_duplicates(subset=["icd_code"])

n_cohort_admissions = int(diagnoses.loc[diagnoses["icd_version"] == 10, "hadm_id"].nunique())

top50 = freq.head(50).merge(d_icd10, on="icd_code", how="left")

table_b3 = pd.DataFrame({
    "Rank":         top50["rank"],
    "ICD-10 Code":  top50["icd_code"].map(eda.format_icd10_code),
    "Description":  top50["long_title"].fillna("(not in d_icd_diagnoses)"),
    "N Admissions": top50["n_admissions"],
    "% of Cohort":  (top50["n_admissions"] / n_cohort_admissions * 100).round(2),
})

pd.set_option("display.max_colwidth", 70)
print(f"Table B3. Top-50 ICD-10 codes (cohort of {n_cohort_admissions:,} ICD-10 admissions)")
print(table_b3.to_string(index=False))

## 8. Codes per admission

Multi-label task sizing. If mean is ~10, we're fitting realistic multi-label signal. If it's 2, multi-label is marginal.

In [ ]:
cpa = eda.codes_per_admission(diagnoses, version=10)
eda.codes_per_admission_stats(cpa)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
eda.plot_codes_per_admission(cpa, ax, max_bin=40)
plt.tight_layout()
plt.savefig(FIGURES / "codes_per_admission.png", bbox_inches="tight")
plt.show()

## 9. Label co-occurrence (top-20 Ã— top-20)

Expected comorbidity patterns â€” e.g., HTN (I10) strongly co-occurs with CHF (I50.9), CKD (N18.*). If the matrix is diagonal-only, something is wrong.

In [ ]:
cooc = eda.label_cooccurrence(diagnoses, top_k=20, version=10)
cooc

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
eda.plot_cooccurrence_heatmap(cooc, ax)
plt.tight_layout()
plt.savefig(FIGURES / "label_cooccurrence.png", bbox_inches="tight")
plt.show()

## 10. Patient demographics + admission patterns

Fairness subgroups for later. Distribution of admissions per patient informs why patient-level splits matter.

In [ ]:
demo = eda.patient_demographics(patients)
demo

In [ ]:
per = eda.admissions_per_patient(admissions)
eda.admissions_per_patient_stats(per)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
# Cap at 20 for readability
ax.hist(per["n_admissions"].clip(upper=20), bins=np.arange(0, 22) - 0.5, color="#4682B4", alpha=0.8)
ax.set_xlabel("Admissions per patient (capped at 20)")
ax.set_ylabel("Patients")
ax.set_title("Admissions-per-patient distribution")
plt.tight_layout()
plt.savefig(FIGURES / "admissions_per_patient.png", bbox_inches="tight")
plt.show()

In [ ]:
# Length of stay + mortality
los = eda.length_of_stay(admissions)
print("Length of stay (days) â€” p50 / mean / p95:")
print(f"  median = {los.median():.2f}")
print(f"  mean   = {los.mean():.2f}")
print(f"  p95    = {los.quantile(0.95):.2f}")

print("\nIn-hospital mortality:")
print(eda.mortality_rate(admissions))

In [ ]:
# Table B4 â€” Demographic summary of the modelable cohort

dx10_hadm   = set(diagnoses.loc[diagnoses["icd_version"] == 10, "hadm_id"].dropna().astype(int))
note_hadm   = set(notes["hadm_id"].dropna().astype(int))
adm_hadm    = set(admissions["hadm_id"].dropna().astype(int))
cohort_hadm = note_hadm & adm_hadm & dx10_hadm

cohort_admissions = admissions[admissions["hadm_id"].isin(cohort_hadm)]
cohort_subjects   = set(cohort_admissions["subject_id"].unique())
cohort_patients   = patients[patients["subject_id"].isin(cohort_subjects)]

demo_cohort              = eda.patient_demographics(cohort_patients)
adm_per_pat_cohort       = eda.admissions_per_patient(cohort_admissions)
adm_per_pat_stats_cohort = eda.admissions_per_patient_stats(adm_per_pat_cohort)

print("Table B4. Demographic summary of the modelable cohort")
print(f"Cohort size: {len(cohort_hadm):,} admissions across {len(cohort_subjects):,} patients")
print()
print("-- Gender and age buckets --")
print(demo_cohort.to_string(index=False))
print()
print("-- Admissions per patient (within cohort) --")
for k, v in adm_per_pat_stats_cohort.items():
    print(f"  {k}: {v}")
print()
print("NOTE: anchor_age is capped at 89 in MIMIC-IV for re-identification protection;")
print("the 85+ bucket compresses all patients aged 89 or older.")


## 11. Join coverage â€” notes âˆ© admissions âˆ© diagnoses

The actual N for the project is `all_three`, not `total_notes`.

In [ ]:
jc = eda.join_coverage(notes, admissions, diagnoses, version=10)
jc

## 12. Version reconciliation â€” v2.2 notes vs v3.1 Hosp

Sanity check on the known version mismatch. Expected loss should be tiny (< 1%).

In [ ]:
# Â§12. Version Reconciliation â€” formatted output for reporting

drift = eda.version_reconciliation(notes, admissions)
dx10 = diagnoses[diagnoses["icd_version"] == 10]
cov = eda.cohort_coverage(notes, admissions, dx10)

# Total loss when joining notes against ICD-10 diagnoses = drift + cohort-filter
total_missing_dx10 = cov["n_drift_loss"] + cov["n_cohort_filter_loss"]
pct_missing_dx10 = round(total_missing_dx10 / cov["n_notes"] * 100, 3)

print(f"Notes missing from v3.1 admissions:    {drift['n_notes_missing_from_hosp']:>7,} ( {drift['pct_notes_missing_from_hosp']:>6.3f} %)")
print(f"Notes missing from v3.1 diagnoses_v10: {total_missing_dx10:>7,} ( {pct_missing_dx10:>6.3f} %)")

# Decomposition â€” honest breakdown so you can disambiguate in the report narrative
print()
print("Breakdown of the 63% loss:")
print(f"  drift (notes absent from v3.1 Hosp entirely): {cov['n_drift_loss']:>7,} ( {cov['pct_drift_loss']:>6.3f} %)")
print(f"  cohort filter (ICD-9 admissions):              {cov['n_cohort_filter_loss']:>7,} ( {cov['pct_cohort_filter_loss']:>6.3f} %)")
